# 01 · Dataset Exploration & EDA
Explore the genuine Kaggle Vehicle Insurance Claim Fraud Detection CSV. Target: FraudFound_P (0 legitimate, 1 fraud). Dataset dimensions and findings are calculated at runtime; no data or results are fabricated.
Source: https://www.kaggle.com/datasets/shivamb/vehicle-claim-fraud-detection


In [ ]:
%pip -q install pandas numpy matplotlib seaborn
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
try:
 from google.colab import drive, files
 drive.mount('/content/drive')
except ImportError: files=None
folder=Path('/content/drive/MyDrive/insurance_fraud_ml/data')
csvs=[p for p in folder.glob('*.csv') if p.name not in ['X_train.csv','X_test.csv','y_train.csv','y_test.csv']] if folder.exists() else []
if csvs: path=csvs[0]
elif files:
 print('Upload the original Kaggle CSV'); up=files.upload()
 if not up: raise FileNotFoundError('No CSV uploaded')
 path=Path(next(iter(up)))
else: path=Path(input('Path to original CSV: '))
if not path.exists(): raise FileNotFoundError(path)
df=pd.read_csv(path); target='FraudFound_P'
if target not in df: raise ValueError(f'Missing {target}. Found {df.columns.tolist()}')
print('Dataset:',df.shape); print('Target counts:',df[target].value_counts(dropna=False).to_dict())

## Inspection and data quality
Inspect actual fields, types, summary statistics, missing values and duplicates before deciding how to model.


In [ ]:
display(df.head()); display(df.tail()); print('Columns:',df.columns.tolist()); display(df.dtypes.astype(str).to_frame('dtype')); display(df.describe(include='all').T); display(df.isna().sum().to_frame('missing')); print('Duplicate records:',df.duplicated().sum())

## Class balance and distributions
Fraud datasets can be imbalanced; accuracy alone can be misleading. Interpret the observed plots and counts.


In [ ]:
sns.set_theme(style='whitegrid'); ax=sns.countplot(data=df,x=target); ax.set(title='Legitimate and fraudulent claims',xlabel='FraudFound_P (0 legitimate, 1 fraud)',ylabel='Records'); plt.show(); display((df[target].value_counts(normalize=True)*100).rename('percent').to_frame())
num=df.select_dtypes(include='number').columns.drop(target,errors='ignore').tolist()[:12]
if num: df[num].hist(figsize=(14,9),bins=25); plt.suptitle('Numeric feature distributions'); plt.tight_layout(); plt.show()
if len(num)>1: sns.heatmap(df[num+[target]].corr(numeric_only=True),cmap='vlag',center=0); plt.title('Numeric feature correlations'); plt.show()
cat=df.select_dtypes(exclude='number').columns.tolist()
for col in cat[:8]:
 vals=df[col].astype('string').fillna('Missing'); top=vals.value_counts().head(10).index
 plt.figure(figsize=(9,3)); sns.countplot(y=vals,order=top); plt.title('Category distribution: '+col); plt.xlabel('Records'); plt.ylabel(col); plt.show()
for col in [c for c in ['Make','AccidentArea','Fault','PolicyType','VehicleCategory','PoliceReportFiled','BasePolicy'] if c in df][:5]:
 d=df.assign(_y=pd.to_numeric(df[target],errors='coerce')).groupby(col,dropna=False)['_y'].mean().sort_values(ascending=False).head(12); d.plot(kind='bar',title='Observed fraud share by '+col); plt.ylabel('Mean target'); plt.show(); print('Descriptive only; small groups can be noisy.')

Interpretation: use the displayed outputs to report the real row count, class proportions, missingness, duplicates and patterns. Correlation and category rates are descriptive, not causal.